# TP/FP/FN/TN Visualization Notebook

This notebook runs `visualize_tp_fp_fn_tn.py` to generate qualitative error panels for the ECNN model.

## 1) Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Running in Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

In [ ]:
# Repo setup (Colab)
REPO_DIR = '/content/symAD-ECNN'
REPO_URL = os.environ.get('SYMAD_REPO_URL', 'https://github.com/RifaDeen/symAD-ECNN.git')

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print('Cloning repo...')
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print('Pulling latest changes...')
        !git -C {REPO_DIR} pull

EVALS_DIR = Path(REPO_DIR) / 'notebooks' / 'evals' if IN_COLAB else Path.cwd().parent
ECNN_DIR = EVALS_DIR / 'ecnn_thresholding'

for p in [str(EVALS_DIR), str(ECNN_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Evals dir:', EVALS_DIR)
print('ECNN dir:', ECNN_DIR)

In [ ]:
# Ensure e2cnn is available
try:
    import e2cnn  # noqa: F401
    print('e2cnn already installed')
except Exception:
    print('Installing e2cnn...')
    !pip install -q e2cnn
    import e2cnn  # noqa: F401
    print('e2cnn installed')

In [ ]:
import os

# 2. Check for the zips
base = "/content/drive/MyDrive/symAD-ECNN/data"
zips = [f"{base}/train_fast.zip", f"{base}/val_fast.zip", f"{base}/test_fast.zip"]

missing = [f for f in zips if not os.path.exists(f)]

if len(missing) == 0:
    print("GOOD NEWS: Zip files found! Proceeding to extraction...")
else:
    print("WARNING: Zip files missing. Please run the CNN-AE notebook first to create them.")
    print(f"   Missing: {missing}")

In [ ]:
# ==========================================
# TURBO LOADER (Unzip to Local)
# ==========================================
import zipfile
import shutil
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/symAD-ECNN/data")
LOCAL_DIR = Path("/content/local_data")

ZIPS_TO_EXTRACT = {
    "train": BASE_DIR / "train_fast.zip",
    "val":   BASE_DIR / "val_fast.zip",
    "test":  BASE_DIR / "test_fast.zip"
}

print("Extracting to Local Disk...")

# Ensure LOCAL_DIR exists
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

for name, zip_path in ZIPS_TO_EXTRACT.items():
    target_dir = LOCAL_DIR / name
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    if zip_path.exists():
        print(f"   Unzipping {name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
    else:
        print(f"   ERROR: {zip_path} not found!")

print("\nData Ready! Local folders created.")

# Update data_paths to point to the locally extracted directories
data_paths["ixi_train"] = LOCAL_DIR / "train"
data_paths["ixi_val"] = LOCAL_DIR / "val"
data_paths["ixi_test"] = LOCAL_DIR / "test"

print("\nUpdated data_paths:")
for key, path in data_paths.items():
    status = "Found" if path and path.exists() else "Not found"
    print(f"  {key}: {status}")
    if path:
        print(f"    -> {path}")

## 2) Run TP/FP/FN/TN Visualization

In [ ]:
from pathlib import Path
from visualize_tp_fp_fn_tn import visualize_tp_fp_fn_tn
from path_utils import find_ecnn_checkpoint, find_data_paths

search_root = Path('/content/symAD-ECNN') if IN_COLAB else None
ecnn_path, ecnn_candidates = find_ecnn_checkpoint(search_root) if search_root else find_ecnn_checkpoint()

if ecnn_path is None:
    raise FileNotFoundError(f'ECNN checkpoint not found. Candidates: {ecnn_candidates[:10]}')

data_paths = find_data_paths(search_root) if search_root else find_data_paths()
normal_path = data_paths.get('ixi_val') or data_paths.get('ixi_test')
anomaly_path = data_paths.get('brats_test')

print('Checkpoint:', ecnn_path)
print('Normal path:', normal_path)
print('Anomaly path:', anomaly_path)

samples = visualize_tp_fp_fn_tn(
    checkpoint_path=ecnn_path,
    normal_data_path=normal_path,
    anomaly_data_path=anomaly_path,
    threshold=None,
    score_method='mean',
    max_samples_per_category=8,
    n_rows_per_panel=4,
    device='cuda'
)

print({k: len(v) for k, v in samples.items()})

In [ ]:
# Display generated panels
from IPython.display import Image, display
from config import FIGURES_DIR

out_dir = FIGURES_DIR / 'tp_fp_fn_tn'
for category in ['tp', 'fp', 'fn', 'tn']:
    img_path = out_dir / f'{category}_samples.png'
    if img_path.exists():
        print(f'\n{category.upper()} samples:')
        display(Image(filename=str(img_path), width=1000))

summary_path = out_dir / 'classification_summary.png'
if summary_path.exists():
    print('\nSummary grid:')
    display(Image(filename=str(summary_path), width=1000))